In [1]:
import pandas as pd
from glob import glob
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import warnings
from tqdm import trange
from time import time
#warnings.filterwarnings("ignore", category=pd.core.common.SettingWithCopyWarning)
#warnings.filterwarnings("ignore")
from scipy.spatial.distance import cdist
from scipy.spatial import distance_matrix
import pdb
from math import sin, cos, sqrt, atan2, radians
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics.pairwise import pairwise_distances
import multiprocessing
from multiprocessing import Pool
from tqdm import tqdm
n_cores = multiprocessing.cpu_count()-200
import os

In [3]:
n_cores

-188

In [ ]:
# set output directory:
# define the output folder path:

output_folder = '../../output_data'

# Check if the folder exists, if not, create it:
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

### load functions from script: functions:

In [4]:
%load_ext autoreload
%autoreload 2
from functions import fundist, proc

In [6]:
#--- Stations Data ---

tic = time()
stations = gpd.read_file('../../output_data/merged_datasets/stations.shp')
stations = stations.set_index(['dataset','site_id']).sort_index()
stations = stations.reset_index()
stations['site_id'] = stations['site_id'].astype(str)
stations = stations.set_index(['dataset','site_id']) 
print('Block time [seg]: ',f'{time()-tic}')

Block time [seg]:  0.8787922859191895


In [8]:
from multiprocessing import Pool
import numpy as np
from time import time as ts

if __name__ == '__main__':
    parallel = True
    xy_coords = np.dstack([stations.geometry.x.values, stations.geometry.y.values])[0]
    data = xy_coords       # YOUR data: np.array[n_samples x m_features]
    n_processes = 10                  # YOUR number of processors
    # dist function: fundist

    # Parameters for parallelization
    n = data.shape[0]
    k_max = n * (n - 1) // 2  # maximum elements in 1D dist array
    k_step = n ** 2 // 500    # ~500 bulks
    dist = np.zeros(k_max)    # resulting 1D dist array
    
    print("Starting ...")
    ts_start = time()
    #From function: start, k_step, k_max, n, data = input_
    inputs = [(k, k_step, k_max, n, data) for k in range(0, k_max, k_step)]
    if parallel:
        with Pool(processes=n_processes) as p:
            max_ = len(inputs)
            with tqdm(total=max_) as pbar:
                for k1, k2, res in p.imap_unordered(proc,inputs):
                    dist[k1:k2] = res
                    pbar.update()

        
        # with Pool(n_processes) as pool:
        #     for k1, k2, res in pool.imap_unordered(proc,inputs):
        #         dist[k1:k2] = res
    else:
        for k in trange(0, k_max, k_step):
            k1,k2,res = proc((k, k_step, k_max, n, data))
            dist[k1:k2] = res
    print("Elapsed %.0f minutes" % ((time() - ts_start) / 60))
    print("Elapsed %.2f seconds" % ((time() - ts_start)))
    print("DONE")
    points = [f'point_{i}' for i in range(1, len(data) + 1)]
    points = stations.index.values
    distmatrix = pd.DataFrame(squareform(dist), columns=points, index=points)
    #print(distmatrix)
    del dist
    print("Elapsed %.2f seconds" % ((time() - ts_start)))


Starting ...


100%|██████████| 250/250 [1:04:34<00:00, 15.50s/it]


Elapsed 65 minutes
Elapsed 3875.86 seconds
DONE


MemoryError: Unable to allocate 40.7 GiB for an array with shape (73924, 73924) and data type float64

In [9]:
# change working directory to output folder:
os.chdir(f'{output_folder}')
tic = time()
import pickle
pickle.dump(distmatrix,open('/distmatrix.pkl','wb'))
print("Elapsed %.2f seconds" % ((time() - ts_start)))


Elapsed 1116.70 seconds


In [10]:
# implement a threshold of 0.5 km, all stations that don't have any neighbour station 
# within 0.5km are not further considered as duplicated stations
thrs = 0.5
# select all columns of stations by adding True, if any distance is smaller threshold of 0.5km
stations_duplicated_bool = ((distmatrix<thrs).sum(axis=1)>1)
print(stations_duplicated_bool)
stations_duplicated_index = stations_duplicated_bool[stations_duplicated_bool].index

(GRQA, 100001)     True
(GRQA, 100002)     True
(GRQA, 100003)     True
(GRQA, 100004)     True
(GRQA, 100005)     True
                  ...  
(sweden, 64)      False
(sweden, 65)      False
(sweden, 66)       True
(sweden, 7)        True
(sweden, 9)       False
Length: 73924, dtype: bool


In [11]:
print(distmatrix.info())
print(stations_duplicated_bool)

<class 'pandas.core.frame.DataFrame'>
Index: 73924 entries, ('GRQA', '100001') to ('sweden', '9')
Columns: 73924 entries, ('GRQA', '100001') to ('sweden', '9')
dtypes: float64(73924)
memory usage: 40.7+ GB
None
(GRQA, 100001)     True
(GRQA, 100002)     True
(GRQA, 100003)     True
(GRQA, 100004)     True
(GRQA, 100005)     True
                  ...  
(sweden, 64)      False
(sweden, 65)      False
(sweden, 66)       True
(sweden, 7)        True
(sweden, 9)       False
Length: 73924, dtype: bool


In [12]:
ts_start = time()
duplicated_dist_matrix = distmatrix[stations_duplicated_bool]
duplicated_dist_matrix = duplicated_dist_matrix.loc[:,stations_duplicated_bool]
print("Elapsed %.2f seconds" % ((time() - ts_start)))

Elapsed 14.58 seconds


In [13]:
ts_start = time()
import pickle
pickle.dump(duplicated_dist_matrix,open('duplicated_only_distmatrix.pkl','wb'))
print("Elapsed %.2f seconds" % ((time() - ts_start)))

Elapsed 16.71 seconds


In [14]:
aux = distmatrix[('GRQA', '100001')]
aux[aux<0.5]

(GRQA, 100001)           0.000
(GRQA, USGS-01015000)    0.291
Name: (GRQA, 100001), dtype: float64